In [1]:
# Core Python libraries
import re
import numpy as np
import pandas as pd
import nltk
from datasets import Dataset

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

# Transformers
from transformers import (
    AutoTokenizer, 
    AutoModel, 
    AutoConfig,
    PreTrainedModel,
    Trainer, 
    TrainingArguments,
    EarlyStoppingCallback
)
from transformers.modeling_outputs import SequenceClassifierOutput

# Scikit-learn
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler  # ← This was missing!
from sklearn.metrics import (
    accuracy_score, 
    f1_score, 
    precision_score, 
    recall_score
)

2025-07-21 18:40:58.401320: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753123258.424242     189 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753123258.431059     189 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
import random
import numpy as np
import torch

def set_seed(seed=2025):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(2025)

In [3]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [4]:
defined_max_len = 400

In [5]:
# Set device
def set_device():
  """
  Set the device. CUDA if available, CPU otherwise

  Args:
    None

  Returns:
    Nothing
  """
  device = "cuda" if torch.cuda.is_available() else "cpu"
  if device != "cuda":
    print("WARNING: For this notebook to perform best, "
        "if possible, in the menu under `Runtime` -> "
        "`Change runtime type.`  select `GPU` ")
  else:
    print("GPU is enabled in this notebook.")

  return device

DEVICE = set_device()

GPU is enabled in this notebook.


In [6]:
# Load the dataset
url = 'https://raw.githubusercontent.com/pinchunc/NMA_DL_SentimentAnalysis/main/data/hippoCorpusV2.csv'
df = pd.read_csv(url)
df = df[['story', 'memType', 'stressful']].dropna() # we exclude n/a spaces
df.head()
df = df[df['memType'] == 'recalled']

In [7]:
# @title Relabel Stress Levels
def relabel_stress(level):
  if level == 1:
    return 0
  else:
    return 1

df['stress_label'] = df['stressful'].apply(relabel_stress)

'''# Plot new grouped stress levels
plt.figure(figsize=(6, 4))
sns.histplot(df['stress_label'], discrete=True, kde=False)
plt.title('Histogram of Grouped Stress Levels')
plt.xlabel('Stress Category')
plt.ylabel('Count')
plt.xticks([0, 1], ['Low', 'High'])
plt.show()

print("Unique values in original 'stressful' column:", df['stressful'].unique())
print("Unique values in new 'stress_label' column:", df['stress_label'].unique())'''

'# Plot new grouped stress levels\nplt.figure(figsize=(6, 4))\nsns.histplot(df[\'stress_label\'], discrete=True, kde=False)\nplt.title(\'Histogram of Grouped Stress Levels\')\nplt.xlabel(\'Stress Category\')\nplt.ylabel(\'Count\')\nplt.xticks([0, 1], [\'Low\', \'High\'])\nplt.show()\n\nprint("Unique values in original \'stressful\' column:", df[\'stressful\'].unique())\nprint("Unique values in new \'stress_label\' column:", df[\'stress_label\'].unique())'

In [8]:
# @title 1) Preprocessing
# Ultra-minimal preprocessing
df['story'] = df['story'].astype(str).fillna('').str.strip()
df['labels'] = df['stress_label'].astype(int)

In [9]:
# @title 3) Create Class Object
# Create an object of class StressDataset to use Hugging Face's 'Trainer'
class StressDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=defined_max_len):
      """
        texts: list or array of input texts (cleaned stories)
        labels: list or array of corresponding stress level labels
        tokenizer: Hugging Face DestilRoberta Tokenizer
        max_len: max length of tokenized input (default = 400 tokens)
      """

      self.texts = texts
      self.labels = labels
      self.tokenizer = tokenizer
      self.max_len = max_len

    def __getitem__(self, idx):
        encoding = self.tokenizer(    # tokenize the text
            self.texts[idx],          # retrieves the story text at index 'idx'
            truncation=True,          # cuts the input off it it exceeds max_len
            padding='max_length',     # pads the input up to max_len tokens
            max_length=self.max_len,  # sets the maximum length of tokens
            return_tensors='pt'       # Returns tensors in PyTorch format
        )

        # This diccionary with token IDs, masks for padding and labels of stress levels
        return {
            'input_ids': encoding['input_ids'].squeeze(),               # token IDs of the story text
            'attention_mask': encoding['attention_mask'].squeeze(),     # binary mask (1s and 0s) telling the model which tokens are content and which are padding.
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)  # retrieves the corresponding label of the story at index 'idx' and converts it into a PyTorch tensor with interger type (long) for classification
        }

    def __len__(self):
        return len(self.texts) # this returns the number of stories in the dataset

In [10]:
class SimpleLIWCExtractor:
    """Simplified LIWC extractor for integration"""
    
    def __init__(self):
        print("Initializing Simple LIWC Extractor...")
        
        # Key psychological categories for stress detection
        self.keyword_dict = {
            'posemo': [
                'happy', 'joy', 'love', 'excellent', 'good', 'great', 'wonderful', 'amazing',
                'fantastic', 'perfect', 'beautiful', 'awesome', 'brilliant', 'delighted',
                'pleased', 'excited', 'cheerful', 'optimistic', 'satisfied', 'grateful',
                'proud', 'confident', 'blessed', 'peaceful', 'content', 'energized'
            ],
            'negemo': [
                'sad', 'angry', 'hate', 'terrible', 'awful', 'horrible', 'bad', 'upset',
                'disgusting', 'annoying', 'disappointing', 'frustrating', 'depressing',
                'miserable', 'furious', 'irritated', 'disturbed', 'worried', 'concerned',
                'negative', 'wrong', 'fail', 'problem', 'difficult', 'hard', 'tough'
            ],
            'anx': [
                'anxious', 'worried', 'nervous', 'stress', 'tension', 'panic', 'fear',
                'afraid', 'scared', 'terrified', 'overwhelmed', 'restless', 'uneasy',
                'apprehensive', 'concerned', 'distressed', 'troubled', 'agitated',
                'pressure', 'strain', 'burden', 'overwhelming', 'stressful', 'crushing'
            ],
            'anger': [
                'angry', 'mad', 'furious', 'rage', 'irritated', 'annoyed', 'frustrated',
                'outraged', 'livid', 'enraged', 'infuriated', 'resentful', 'bitter',
                'hostile', 'aggressive', 'violent', 'hate', 'despise', 'loathe'
            ],
            'sad': [
                'sad', 'depressed', 'unhappy', 'miserable', 'down', 'blue', 'crying',
                'tears', 'weeping', 'grief', 'sorrow', 'melancholy', 'gloomy',
                'heartbroken', 'devastated', 'disappointed', 'dejected', 'despondent',
                'lonely', 'empty', 'hopeless', 'despair', 'exhausted', 'burned'
            ],
            'work': [
                'work', 'job', 'career', 'office', 'business', 'employee', 'boss',
                'company', 'organization', 'professional', 'workplace', 'deadline',
                'project', 'task', 'meeting', 'supervisor', 'colleague', 'performance'
            ],
            'family': [
                'family', 'mother', 'father', 'parent', 'child', 'sister', 'brother',
                'mom', 'dad', 'son', 'daughter', 'home', 'grandmother', 'grandfather',
                'spouse', 'husband', 'wife', 'kids', 'children'
            ],
            'friend': [
                'friend', 'buddy', 'pal', 'companion', 'colleague', 'mate',
                'friendship', 'classmate', 'roommate', 'neighbor', 'partner',
                'friends', 'conversation', 'social'
            ],
            'achieve': [
                'achieve', 'success', 'goal', 'accomplish', 'win', 'complete', 'finish',
                'succeed', 'victory', 'triumph', 'attain', 'reach', 'obtain',
                'fulfill', 'realize', 'master', 'excel', 'outstanding', 'award',
                'proud', 'accomplished', 'productive'
            ],
            'focuspast': [
                'was', 'were', 'had', 'did', 'yesterday', 'ago', 'before', 'previous',
                'earlier', 'formerly', 'once', 'used', 'past', 'remember', 'recalled'
            ],
            'focusfuture': [
                'will', 'going', 'tomorrow', 'next', 'future', 'plan', 'hope',
                'expect', 'anticipate', 'predict', 'upcoming', 'later', 'eventually',
                'planning', 'excited', 'look'
            ],
            'certain': [
                'always', 'never', 'definitely', 'sure', 'certain', 'absolutely',
                'completely', 'totally', 'obviously', 'clearly', 'undoubtedly',
                'confident', 'exactly', 'precisely'
            ],
            'tentat': [
                'maybe', 'perhaps', 'might', 'could', 'possibly', 'probably',
                'seems', 'appears', 'likely', 'potentially', 'suppose', 'guess',
                'uncertain', 'unclear', 'doubt', 'question'
            ]
        }
        
        self.categories = list(self.keyword_dict.keys())
        self.scaler = StandardScaler()
        self.fitted = False
    
    def extract_features(self, text):
        """Extract LIWC features from text"""
        if pd.isna(text):
            text = ""
        
        text_lower = text.lower()
        words = re.findall(r'\b\w+\b', text_lower)
        total_words = len(words) if words else 1
        
        # Extract LIWC features
        features = []
        for category in self.categories:
            keywords = self.keyword_dict[category]
            count = sum(1 for word in words if word in keywords)
            percentage = (count / total_words) * 100
            features.append(percentage)
        
        # Add basic text statistics
        features.extend([
            len(words),                                    # word_count
            len(text),                                     # char_count
            text.count('!'),                              # exclamation_count
            text.count('?'),                              # question_count
            sum(1 for c in text if c.isupper()) / len(text) if text else 0,  # caps_ratio
        ])
        
        return np.array(features)
    
    def extract_batch_features(self, texts):
        """Extract features for a batch of texts"""
        features = []
        for text in texts:
            features.append(self.extract_features(text))
        return np.array(features)
    
    def fit_transform(self, texts):
        """Fit scaler and transform features"""
        features = self.extract_batch_features(texts)
        scaled_features = self.scaler.fit_transform(features)
        self.fitted = True
        print(f"✓ LIWC features: {features.shape[1]} per text")
        return scaled_features
    
    def transform(self, texts):
        """Transform features using fitted scaler"""
        if not self.fitted:
            raise ValueError("Must fit extractor first")
        features = self.extract_batch_features(texts)
        return self.scaler.transform(features)
    
    def get_feature_count(self):
        """Get number of features extracted"""
        return len(self.categories) + 5  # LIWC categories + 5 basic stats

In [11]:
# @title 4) Split the dataset (training, validation and testing) with LIWC integration
from sklearn.model_selection import train_test_split

# Initialize LIWC extractor
liwc_extractor = SimpleLIWCExtractor()

# First split: Separate test set (10%)
train_val_texts, test_texts, train_val_labels, test_labels = train_test_split(
    df['story'].tolist(),
    df['labels'].tolist(),
    test_size=0.1,  # 10% for testing
    random_state=42,
    stratify=df['labels']
)

# Second split: Split remaining 90% into train (80%) and validation (10%)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_val_texts,
    train_val_labels,
    test_size=0.111,  # 10% of total = 0.1/0.9 ≈ 0.111 of remaining
    random_state=42,
    stratify=train_val_labels
)

# Print split information
total_samples = len(df)
print(f"Total samples: {total_samples}")
print(f"Training: {len(train_texts)} ({len(train_texts)/total_samples*100:.1f}%)")
print(f"Validation: {len(val_texts)} ({len(val_texts)/total_samples*100:.1f}%)")
print(f"Testing: {len(test_texts)} ({len(test_texts)/total_samples*100:.1f}%)")

# Extract and fit LIWC features on training data
print("\nExtracting LIWC features...")
train_liwc_features = liwc_extractor.fit_transform(train_texts)
val_liwc_features = liwc_extractor.transform(val_texts)
test_liwc_features = liwc_extractor.transform(test_texts)

print(f"LIWC feature dimensions: {train_liwc_features.shape[1]}")

# Verify class distribution in each split
print(f"\nClass distribution:")
train_class_dist = np.bincount(train_labels)
val_class_dist = np.bincount(val_labels)
test_class_dist = np.bincount(test_labels)

print(f"Training - Class 0: {train_class_dist[0]}, Class 1: {train_class_dist[1]}")
print(f"Validation - Class 0: {val_class_dist[0]}, Class 1: {val_class_dist[1]}")
print(f"Testing - Class 0: {test_class_dist[0]}, Class 1: {test_class_dist[1]}")

# Enhanced Dataset creation with LIWC features and new tokenization approach
# Create base datasets
train_dataset = Dataset.from_dict({
    'text': train_texts, 
    'labels': train_labels,
    'liwc_features': train_liwc_features.tolist()  # Convert numpy to list for datasets
})
val_dataset = Dataset.from_dict({
    'text': val_texts, 
    'labels': val_labels,
    'liwc_features': val_liwc_features.tolist()
})
test_dataset = Dataset.from_dict({
    'text': test_texts, 
    'labels': test_labels,
    'liwc_features': test_liwc_features.tolist()
})

# Initialize model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")

# Tokenization function that preserves LIWC features
def tokenize_function(examples):
    tokenized = tokenizer(
        examples['text'], 
        truncation=True, 
        padding=False, 
        max_length=defined_max_len  # Shorter for CPU
    )
    # Preserve LIWC features and labels
    tokenized['liwc_features'] = examples['liwc_features']
    tokenized['labels'] = examples['labels']
    return tokenized

# Apply tokenization while preserving LIWC features
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("✓ Datasets tokenized with LIWC features preserved")
print(f"Training dataset features: {train_dataset.column_names}")
print(f"Sample LIWC features shape: {len(train_dataset[0]['liwc_features'])}")

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=[0, 1],
    y=train_labels
)
weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print("Class weights:", weights)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha  # class weights
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        if self.alpha is not None and not isinstance(self.alpha, torch.Tensor):
            self.alpha = torch.tensor(self.alpha, dtype=torch.float, device=DEVICE)
                     
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)  # predicted prob of the true class
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss


Initializing Simple LIWC Extractor...
Total samples: 2779
Training: 2223 (80.0%)
Validation: 278 (10.0%)
Testing: 278 (10.0%)

Extracting LIWC features...
✓ LIWC features: 18 per text
LIWC feature dimensions: 18

Class distribution:
Training - Class 0: 915, Class 1: 1308
Validation - Class 0: 114, Class 1: 164
Testing - Class 0: 114, Class 1: 164


Map:   0%|          | 0/2223 [00:00<?, ? examples/s]

Map:   0%|          | 0/278 [00:00<?, ? examples/s]

Map:   0%|          | 0/278 [00:00<?, ? examples/s]

✓ Datasets tokenized with LIWC features preserved
Training dataset features: ['text', 'labels', 'liwc_features', 'input_ids', 'attention_mask']
Sample LIWC features shape: 18
Class weights: tensor([1.2148, 0.8498], device='cuda:0')


In [12]:
# @title 5) Simplified Enhanced RobertaClassifier (Similar to Standard + LIWC)
class SimplifiedEnhancedRobertaClassifier(PreTrainedModel):
    def __init__(self, config, liwc_feature_dim, class_weights=None):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.class_weights = class_weights
        self.liwc_feature_dim = liwc_feature_dim
        
        # RoBERTa base model (same as standard)
        self.roberta = AutoModel.from_pretrained("FacebookAI/roberta-base", config=config)
        self.hidden_size = config.hidden_size  # 768
        
        # LIWC feature processor (simple, like standard model)
        self.liwc_processor = nn.Sequential(
            nn.Linear(self.liwc_feature_dim, 64),
            nn.Tanh(),  # Same activation as standard model
            nn.Dropout(config.hidden_dropout_prob),  # Same dropout as standard
            nn.Linear(64, 64)  # Keep 64 LIWC features
        )
        
        # Combined classification head (similar to RobertaClassificationHead)
        combined_dim = self.hidden_size + 64  # 768 (CLS) + 64 (LIWC) = 832
        
        self.classification_head = nn.Sequential(
            nn.Dropout(config.hidden_dropout_prob),  # Same as standard
            nn.Linear(combined_dim, combined_dim),    # 832 → 832 (like standard 768→768)
            nn.Tanh(),                                # Same activation as standard
            nn.Dropout(config.hidden_dropout_prob),  # Same as standard
            nn.Linear(combined_dim, self.num_labels) # 832 → 2 (final output)
        )

    def forward(self, input_ids, attention_mask=None, liwc_features=None, labels=None):
        # 1. RoBERTa encoding (exactly like standard model)
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        
        # 2. Take [CLS] token representation (exactly like standard model)
        sequence_output = outputs.last_hidden_state
        pooled_output = sequence_output[:, 0]  # [CLS] token, shape: (batch_size, 768)
        
        # 3. Process LIWC features (simple processing)
        liwc_processed = self.liwc_processor(liwc_features)  # (batch_size, 64)
        
        # 4. Combine [CLS] token with LIWC features
        combined_features = torch.cat([pooled_output, liwc_processed], dim=1)  # (batch_size, 832)
        
        # 5. Classification head (similar to standard model)
        logits = self.classification_head(combined_features)
        
        # 6. Loss calculation
        loss = None
        if labels is not None:
            if self.class_weights is not None:
                loss_fn = FocalLoss(alpha=self.class_weights, gamma=2.0)
                loss = loss_fn(logits, labels)
            else:
                loss = F.cross_entropy(logits, labels)  # Standard cross-entropy like original
        
        return SequenceClassifierOutput(
            loss=loss,
            logits=logits
        )


In [13]:
# @title 6) Initialize Enhanced Model with same base model
config = AutoConfig.from_pretrained(
    'FacebookAI/roberta-base',  # ← Updated to use same model path
    num_labels=2,
)

# Get LIWC feature dimension
liwc_feature_dim = liwc_extractor.get_feature_count()
print(f"LIWC feature dimension: {liwc_feature_dim}")

model = SimplifiedEnhancedRobertaClassifier(config, liwc_feature_dim, class_weights=weights)
model = model.to(DEVICE)  # ← Added explicit device transfer like your original

print(f"✓ Model loaded on device: {next(model.parameters()).device}")
print(f"✓ Using same base model: FacebookAI/roberta-base")

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
         
    # More detailed monitoring
    unique_preds, pred_counts = np.unique(preds, return_counts=True)
    unique_labels, label_counts = np.unique(labels, return_counts=True)
         
    print(f"Predicted classes: {dict(zip(unique_preds, pred_counts))}")
    print(f"True classes: {dict(zip(unique_labels, label_counts))}")
         
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro'),
        'f1_weighted': f1_score(labels, preds, average='weighted'),
        'precision': precision_score(labels, preds, average='macro', zero_division=0),
        'recall': recall_score(labels, preds, average='macro', zero_division=0)
    }

# @title 7) Enhanced Data Collator for LIWC Features with new dataset format
from transformers import DataCollatorWithPadding

class DataCollatorWithLIWC:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.base_collator = DataCollatorWithPadding(tokenizer)
    
    def __call__(self, features):
        # Separate LIWC features and labels from tokenized features
        liwc_features = [f['liwc_features'] for f in features]
        labels = [f['labels'] for f in features]
        
        # Remove LIWC features and labels from tokenized features for base collator
        tokenized_features = []
        for f in features:
            tokenized_f = {k: v for k, v in f.items() if k not in ['liwc_features', 'labels']}
            tokenized_features.append(tokenized_f)
        
        # Use base collator for tokenized features (handles padding)
        batch = self.base_collator(tokenized_features)
        
        # Add LIWC features and labels back to batch
        batch['liwc_features'] = torch.tensor(liwc_features, dtype=torch.float)
        batch['labels'] = torch.tensor(labels, dtype=torch.long)
        
        return batch

# @title 8) Training Arguments (same as your original)
training_args = TrainingArguments(
    output_dir='./results',
    run_name="stress-classifier-liwc-v1", # Different from output_dir to avoid warning
    logging_dir='./logs',
    report_to='none',

    # Training schedule (balanced approach)
    num_train_epochs=8,                   # Compromise: more training than target's 2
    learning_rate=5e-5,                   # Slightly higher for larger batch size
    
    # Batch sizes (compromise between memory and speed)
    per_device_train_batch_size=32,       # Larger than current 16, smaller than target 64
    per_device_eval_batch_size=32,        # Match training batch size
    
    # Regularization and stability
    weight_decay=0.01,                    # Same as both configs
    warmup_steps=150,                     # Between current 200 and target 100
    max_grad_norm=1.0,                    # Keep for stability (missing in target)
    
    # Evaluation and saving (frequent monitoring)
    eval_strategy="steps",
    eval_steps=20,                        # Slightly less frequent than current 10
    save_strategy="steps",
    save_steps=20,                        # Match eval_steps
    save_total_limit=1,                   # Keep only best model (saves space)
    
    # Model selection
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    
    # Logging
    logging_strategy="steps",
    logging_steps=20,                     # Keep frequent logging
    
    # Performance (from target config)
    dataloader_pin_memory=False,          # Added from target
)

# @title 9) Enhanced Trainer with LIWC Data Collator
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithLIWC(tokenizer),  # NEW: Custom data collator
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
)

LIWC feature dimension: 18


Some weights of RobertaModel were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Model loaded on device: cuda:0
✓ Using same base model: FacebookAI/roberta-base


/tmp/ipykernel_189/1286631591.py:105: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [14]:
# @title 10) Training
print("Starting training with Simplified Enhanced RobertaClassifier + LIWC features...")
print(f"Model architecture:")
print(f"- Text features: [CLS] token (768 dims)")
print(f"- LIWC features: {liwc_feature_dim} → 64") 
print(f"- Combined: {768 + 64} → {768 + 64} → 2 classes")
print(f"- Total parameters: {sum(p.numel() for p in model.parameters()):,}")

trainer.train()

Starting training with Simplified Enhanced RobertaClassifier + LIWC features...
Model architecture:
- Text features: [CLS] token (768 dims)
- LIWC features: 18 → 64
- Combined: 832 → 832 → 2 classes
- Total parameters: 125,345,730


Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 150.00 MiB. GPU 0 has a total capacity of 15.89 GiB of which 133.12 MiB is free. Process 52068 has 15.76 GiB memory in use. Of the allocated memory 14.95 GiB is allocated by PyTorch, and 518.89 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# @title 11) Evaluation
print("Evaluating on test set...")
test_results = trainer.evaluate(test_dataset)
print("Test Results:", test_results)

In [ ]:
import matplotlib.pyplot as plt

# Get training history
logs = trainer.state.log_history
logs

# Extract training and validation data
train_loss = []
train_accuracy = []
val_loss = []
val_accuracy = []
epochs = []
steps = []

for log in logs:
    if 'loss' in log:  # End of epoch training loss
        train_loss.append(log['loss'])
        # train_accuracy.append(log['train_accuracy'])
    if 'eval_loss' in log:   # Validation loss
        val_loss.append(log['eval_loss'])
        val_accuracy.append(log['eval_accuracy'])
        epochs.append(log['epoch'])
        steps.append(log['step'])

In [ ]:
len(val_loss)

In [ ]:
# Create simple plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Loss
if len(train_loss) > 0 and len(val_loss) > 0:
    ax1.plot(steps[1:], train_loss, 'b-o', label='Training Loss')
    ax1.plot(steps[1:], val_loss[:-1], 'r-o', label='Validation Loss')
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.legend()
    ax1.grid(True)

# Plot 2: Validation Accuracy
if len(val_accuracy) > 0:
    ax2.plot(steps, val_accuracy, 'g-o', label='Validation Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Validation Accuracy')
    ax2.legend()
    ax2.grid(True)

plt.tight_layout()
plt.show()